# Lekcia 13 - Pamäť agenta


## Nastavenie

Tento zápisník demonštruje, ako vytvoriť agenta na rezerváciu ciest s **perzistentnou pamäťou** pomocou **Microsoft Agent Framework** (MAF).

Naučíte sa, ako rôzne typy agentovej pamäte — pracovná, krátkodobá a dlhodobá — ovplyvňujú, ako agent uchováva a používa informácie počas rozhovorov.

**Predpoklady:**
- Projekt Microsoft Foundry s nasadeným chatovacím modelom (napr. `gpt-5-mini`).
- Prihlásený cez Azure CLI — spustite `az login` vo vašom termináli.
- `AZURE_AI_PROJECT_ENDPOINT` — endpoint vášho projektu Microsoft Foundry.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — názov vášho nasadeného modelu.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import json
import dotenv
from typing import Annotated
from datetime import datetime

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## Typy pamäte agenta

AI agenti môžu využívať rôzne druhy pamäte, z ktorých každý slúži na odlišný účel:

### Pracovná pamäť
Samotná vlákno konverzácie — správy vymenené počas jednej relácie. Agent sa môže odvolávať na skoršie správy v tom istom vlákne, aby udržal koherenciu. V MAF je to vytvorené pomocou **`agent.create_session()`**, ktorá vráti `AgentSession`.

### Krátkodobá pamäť
Informácie, ktoré pretrvávajú počas trvania úlohy alebo relácie, ale nie sú trvalo uložené. Napríklad agent môže počas viackolovej plánovacej konverzácie zhromažďovať fakty a použiť ich na vytvorenie finálneho itinerára.

### Dlhodobá pamäť
Preferencie a fakty, ktoré pretrvávajú **naprieč reláciami**. Návratový používateľ by nemal musieť opakovane zdieľať svoje diétny obmedzenia alebo štýl cestovania. Dlhodobá pamäť je zvyčajne podporená externým úložiskom — databázou, súborom alebo vektorovým indexom — a prezentovaná agentovi cez nástroje.


## Pracovná pamäť so sedením

Najjednoduchšia forma pamäte je konverzačné sedenie. Keď odovzdáte tú istú session (vytvorenú pomocou `agent.create_session()`) po sebe idúcim volaniam `agent.run()`, agent vidí celú históriu tejto konverzácie a dokáže si vybaviť skoršie detaily.

Vytvorme cestovného agenta a predveďme pracovnú pamäť.


In [ ]:
agent = client.as_agent(
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

Agent si správne vybavil rozpočet, pretože obe správy zdieľajú tú istú reláciu. Toto je **pracovná pamäť** — existuje iba počas životnosti relácie.

### Čo sa stane s novým vláknom?

Ak vytvoríme **novú** reláciu, agent nemá žiadnu pamäť predchádzajúceho rozhovoru:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Vzor dlhodobej pamäte

Aby sme si zapamätali preferencie používateľa **naprieč reláciami**, potrebujeme perzistentné úložisko, ktoré žije mimo vlákna konverzácie. Agent pristupuje k tomuto úložisku cez **nástroje** — funkcie, ktoré môže volať na uloženie a načítanie informácií.

Nižšie implementujeme jednoduché pamäťové úložisko preferencií (v produkcii by ste ho podporili databázou alebo vektorovým indexom) a sprístupníme ho ako nástroje, ktoré agent môže používať.

### Architektúra
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Scenár 1 — Používateľ prvýkrát rezervuje výročnú cestu

Sarah navštívi po prvýkrát. Agent by mal uložiť jej preferencie pomocou nástrojov a použiť ich na odporúčanie hotelov.


In [ ]:
travel_agent = client.as_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Scenár 2 — Sarah sa vracia o niekoľko týždňov neskôr

Sarah začína **úplne novú konverzáciu** (simulujúc novú reláciu). Pracovná pamäť je prázdna, no dlhodobé úložisko preferencií stále obsahuje jej informácie. Agent by ich mal vyhľadať a použiť na personalizovanie odporúčaní.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Zhrnutie

V tejto lekcii ste sa naučili tri typy pamäte agenta a ako ich implementovať pomocou Microsoft Agent Frameworku:

| Typ pamäte | Mechanizmus MAF | Životnosť |
|---|---|---|
| **Pracovná** | `agent.create_session()` | Jedna konverzácia |
| **Krátkodobá** | Nahromadený kontext v rámci vlákna | Jedna úloha / relácia |
| **Dlhodobá** | Externé úložisko prístupné cez funkcie `@tool` | Naprieč reláciami |

### Kľúčové zistenia
1. **`agent.create_session()`** poskytuje pracovnú pamäť — agent vidí celú históriu konverzácie v rámci relácie.
2. **Nové relácie strácajú kontext** — bez dlhodobej pamäte si agent nepamätá minulé rozhovory.
3. **Funkcie `@tool`** prekonávajú túto medzeru — umožňujú agentovi ukladať a získavať informácie z trvalého úložiska.
4. **Personalizácia sa časom zlepšuje** — čím viac preferencií sa uloží, tým lepšie sú odporúčania agenta.

### Praktické využitie
- **Zákaznícky servis**: Pamätať si históriu a preferencie zákazníka
- **Osobní asistenti**: Zachovať kontext počas dní alebo týždňov
- **Zdravotníctvo**: Sledovať informácie a preferencie pacienta
- **E-commerce**: Personalizované nakupovanie na základe histórie

### Ďalšie kroky
- Nahradiť slovník v pamäti databázou alebo vektorovým úložiskom (napr. Azure AI Search)
- Pridať expiráciu pamäte pre časovo citlivé informácie
- Vytvoriť multi-agentné systémy so zdieľanou pamäťou
- Preskúmať Cognee notebook pre pamäť podporovanú vedomostnými grafmi


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vyhlásenie o zodpovednosti**:
Tento dokument bol preložený pomocou AI prekladateľskej služby [Co-op Translator](https://github.com/Azure/co-op-translator). Hoci sa snažíme o presnosť, vezmite prosím na vedomie, že automatické preklady môžu obsahovať chyby alebo nepresnosti. Pôvodný dokument v jeho natívnom jazyku by mal byť považovaný za autoritatívny zdroj. Pre kritické informácie sa odporúča profesionálny ľudský preklad. Nie sme zodpovední za žiadne nedorozumenia alebo nesprávne interpretácie vyplývajúce z použitia tohto prekladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
